# 4. Les LLM sans entraînement (*zero-shot*)

**Objectif** : classer les avis avec un **LLM**, **sans aucun entraînement**, puis le comparer à TF-IDF (93,8 %) et CamemBERT (95,8 %) sur trois critères : **précision, vitesse, coût**.

### Zero-shot, c'est quoi ?
| Approche | Ce qu'on fait | Exemples d'avis vus |
|---|---|---|
| TF-IDF + régression logistique | on entraîne un modèle de zéro | 160 000 |
| CamemBERT | on *fine-tune* un modèle pré-entraîné | 20 000 |
| **LLM zero-shot** | on **décrit la tâche en français** dans un message (le *prompt*), sans exemple | **0** |

Un LLM (*Large Language Model*) a été entraîné sur une énorme quantité de textes : il sait déjà ce qu'est une critique de film. On lui demande simplement « cet avis est-il positif ou négatif ? ». *Zero-shot* = « zéro exemple ». (Si on lui donnait quelques exemples corrigés dans le prompt, ce serait du *few-shot*.)

### Deux LLM très différents
| | **Qwen 2.5 (3B)** via Ollama | **Claude** via l'API d'Anthropic (optionnel, 4.7) |
|---|---|---|
| Où il tourne | sur **ton PC** | sur les serveurs d'Anthropic |
| Taille | 3 milliards de paramètres | beaucoup plus grand |
| Coût | gratuit | payant, au token |
| Confidentialité | rien ne sort de ton PC | les avis sont envoyés à l'API |

C'est un vrai choix d'ingénieur : on comparera les deux.

## 4.0 Vérifier Ollama

Ollama doit être lancé (tu l'as déjà utilisé pour le RAG). La cellule ci-dessous liste les modèles installés : `qwen2.5:3b` doit apparaître.

In [ ]:
import json
import time

import numpy as np
import ollama
import pandas as pd

print([m.model for m in ollama.list().models])

## 4.1 Les données

On tire **500 avis au hasard** dans la validation (toujours les mêmes grâce à `random_state=42`). C'est assez pour mesurer, et le même échantillon servira pour Claude : la comparaison sera juste.

Avec 500 avis, l'accuracy sera moins précise qu'avec 20 000 : tu calculeras cette marge d'erreur au 4.4.

In [ ]:
val = pd.read_parquet("data/validation.parquet")
echantillon = val.sample(500, random_state=42).reset_index(drop=True)

print(echantillon.shape)
print(echantillon["label"].value_counts())

## 4.2 Le prompt et le premier appel

Un appel à un LLM de type « chat » contient :
- un **message système** : la consigne générale, le « rôle » ;
- un **message utilisateur** : ici, l'avis à classer.

Deux réglages importants :

| Réglage | Valeur | Pourquoi |
|---|---|---|
| `format` | un schéma JSON | **sorties structurées** : le modèle est *obligé* de répondre `{"sentiment": "positif"}` ou `{"sentiment": "negatif"}`. Sans ça, il pourrait répondre « Cet avis me semble plutôt positif… », difficile à exploiter automatiquement. |
| `temperature` | 0 | un LLM choisit chaque mot avec une part de hasard ; à 0, il prend toujours le plus probable : **mêmes entrées → mêmes réponses**, c'est reproductible. |

⏱️ **Le premier appel est lent (~30 s)** : Ollama charge le modèle en mémoire. Les suivants prennent moins d'une seconde.

La réponse contient aussi des **compteurs de tokens** : `prompt_eval_count` (tokens lus : consigne + avis) et `eval_count` (tokens écrits). Ici c'est gratuit, mais avec une API payante, c'est ce qu'on paie.

In [ ]:
MODELE_LOCAL = "qwen2.5:3b"

CONSIGNE = (
    "Tu classes des critiques de films en français, publiées sur Allociné. "
    "Indique si l'avis global de l'auteur sur le film est positif ou négatif. "
    "Juge le verdict de l'auteur dans son ensemble, pas des phrases isolées."
)

SCHEMA = {
    "type": "object",
    "properties": {"sentiment": {"type": "string", "enum": ["positif", "negatif"]}},
    "required": ["sentiment"],
    "additionalProperties": False,
}


def appeler_ollama(texte):
    """Envoie un avis au modèle local et renvoie la réponse complète."""
    return ollama.chat(
        model=MODELE_LOCAL,
        messages=[
            {"role": "system", "content": CONSIGNE},
            {"role": "user", "content": texte},
        ],
        format=SCHEMA,
        options={"temperature": 0},
    )


avis = echantillon.loc[0, "review"]
print(avis[:300], "...")
print("Vraie réponse :", echantillon.loc[0, "label"])

reponse = appeler_ollama(avis)
print()
print("Texte renvoyé :", reponse.message.content)
print(reponse.prompt_eval_count, "tokens lus,", reponse.eval_count, "tokens écrits")

### À TOI (4.2)
1. `reponse.message.content` est du **texte** qui ressemble à un dictionnaire. Transforme-le en vrai dictionnaire Python avec `json.loads(...)`, puis récupère la valeur de `"sentiment"`.
2. Convertis-la en **0 ou 1**, comme la colonne `label` (1 = positif).
3. Relance la cellule d'appel une 2ᵉ fois : combien de temps prend-elle maintenant ? Pourquoi le 1ᵉʳ appel était-il si lent ?

In [ ]:
# À TOI


**Mes réponses (4.2)** : ...

## 4.3 La fonction `classer` et un essai sur 20 avis

Avant de lancer une longue boucle, on fait toujours un **essai pilote** sur quelques exemples : ça vérifie que tout marche et ça permet d'estimer la durée totale. (Avec une API payante, ça permet aussi d'estimer le coût **avant** de dépenser : tu le feras au 4.7.)

### À TOI (4.3)
1. Écris une fonction `classer(texte)` qui renvoie un **dictionnaire** :
   ```python
   {"prediction": 1 ou 0, "secondes": ..., "tokens_entree": ..., "tokens_sortie": ...}
   ```
   Pour mesurer le temps : `debut = time.perf_counter()` avant l'appel, puis `time.perf_counter() - debut` après.
2. Applique-la aux **20 premiers avis** de `echantillon` avec une boucle, et range les résultats dans un DataFrame : `pd.DataFrame(liste_de_dictionnaires)`.
3. Affiche l'accuracy sur ces 20 avis et la latence moyenne.
4. **Estime** la durée pour 500 avis, puis pour 20 000.

*Rappel* : une proportion se calcule comme au 3.3, avec `(... == ...).mean()`.

In [ ]:
# À TOI


**Mes estimations (4.3)** : ...

## 4.4 L'évaluation sur 500 avis

### À TOI (4.4)
1. Même boucle qu'au 4.3, sur tout `echantillon` (≈ 5 minutes). Pour voir l'avancement, remplace `range(...)` par `tqdm(range(...))` (`from tqdm.auto import tqdm`).
2. **Sauvegarde** les résultats : `resultats.to_csv("data/qwen_500.csv", index=False)`. Si le noyau redémarre, tu relis le fichier avec `pd.read_csv(...)` au lieu de tout relancer.
3. Calcule l'accuracy et la latence moyenne.
4. **La marge d'erreur** : avec seulement 500 avis, l'accuracy mesurée est approximative. Un intervalle de confiance à 95 % vaut environ $p \pm 1{,}96\sqrt{\dfrac{p(1-p)}{n}}$, avec $p$ l'accuracy et $n = 500$. Calcule-le. Qwen est-il **vraiment** moins bon (ou meilleur) que CamemBERT (95,8 %) ?

In [ ]:
# À TOI


**Ma conclusion (4.4)** : ...

## 4.5 Les erreurs du modèle

Les chiffres ne disent pas tout. Regarde **les avis où Qwen s'est trompé**.

### À TOI (4.5)
1. Ajoute les colonnes de `resultats` à `echantillon` (`pd.concat([echantillon, resultats], axis=1)`), puis filtre les lignes où `prediction != label`.
2. Ses erreurs sont-elles équilibrées ? Compte combien sont des **faux positifs** (avis négatif classé positif) et combien des **faux négatifs**.
3. Lis au moins 5 de ces avis en entier. Pour chacun : est-ce **le modèle qui se trompe**, ou l'**étiquette qui est discutable** (avis mitigé, note qui ne correspond pas au texte) ?

Ce travail d'analyse est très apprécié en entretien : il montre que tu ne fais pas confiance aveuglément à une métrique.

In [ ]:
# À TOI


**Mon analyse (4.5)** : ...

## 4.6 Tes avis pièges

### À TOI (4.6)
Passe tes 3 avis pièges dans `classer` et complète le tableau. (Le LLM donne une réponse, pas une probabilité : écris simplement sa prédiction.)

In [ ]:
# À TOI


| Avis | Vraie réponse | TF-IDF + bigrammes | CamemBERT | Qwen 2.5 3B |
|---|---|---|---|---|
| Piège 1 (« génial ») | négatif | 22 % → négatif ✅ | 0,6 % → négatif ✅ | ... |
| Piège 2 (« pas mauvais ») | positif | 38 % → négatif ❌ | 97,8 % → positif ✅ | ... |
| Piège 3 (inventé) | positif | 0,5 % → négatif ❌ | 97,8 % → positif ✅ | ... |

## 4.7 (Optionnel) Claude via l'API d'Anthropic

À faire **quand tu auras une clé API** (console.anthropic.com → Billing → crédits → API Keys). Un abonnement Claude.ai ne suffit pas : l'API est facturée à part.

### Préparer la clé
1. Dans le dossier `avis-allocine`, crée un fichier nommé exactement **`.env`** contenant une seule ligne : `ANTHROPIC_API_KEY=ta-clé-ici`
2. **Ne colle jamais ta clé dans le notebook** : le notebook ira sur GitHub, et une clé publiée peut être utilisée par n'importe qui à tes frais. Le `.env`, lui, est dans le `.gitignore`.

### Ce qui change par rapport à Ollama
| | Ollama | Claude |
|---|---|---|
| Format imposé | `format=SCHEMA` | `output_config={"format": {"type": "json_schema", "schema": SCHEMA}}` |
| Réponse | `reponse.message.content` | une **liste de blocs** `reponse.content` : un bloc `thinking` (sa réflexion) puis un bloc `text` (le JSON) |
| Tokens | `prompt_eval_count`, `eval_count` | `reponse.usage.input_tokens`, `reponse.usage.output_tokens` |
| Coût | 0 | **4 $ / million** de tokens d'entrée, **20 $ / million** en sortie (réflexion comprise) |

`effort: "low"` demande à Claude de peu réfléchir (la tâche est simple : moins cher, plus rapide). `fallbacks="default"` : si Claude refuse un avis (rare, ses filtres de sécurité peuvent se tromper), un autre modèle prend le relais.

In [ ]:
import os

import anthropic
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("ANTHROPIC_API_KEY"):
    print("⚠️ Pas de clé : crée le fichier .env, puis redémarre le noyau")

client = anthropic.Anthropic()

MODELE_CLAUDE = "claude-opus-5-5"
PRIX_ENTREE = 4 / 1_000_000   # dollars par token d'entrée
PRIX_SORTIE = 20 / 1_000_000  # dollars par token de sortie


def appeler_claude(texte):
    """Envoie un avis à Claude et renvoie la réponse complète de l'API."""
    return client.beta.messages.create(
        model=MODELE_CLAUDE,
        max_tokens=2000,
        system=CONSIGNE,
        messages=[{"role": "user", "content": texte}],
        output_config={"effort": "low", "format": {"type": "json_schema", "schema": SCHEMA}},
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
    )


reponse_c = appeler_claude(echantillon.loc[0, "review"])
print("stop_reason :", reponse_c.stop_reason)
for bloc in reponse_c.content:
    print(bloc.type, "→", getattr(bloc, "text", ""))
print(reponse_c.usage.input_tokens, "tokens d'entrée,", reponse_c.usage.output_tokens, "tokens de sortie")

### À TOI (4.7)
1. Écris `classer_claude(texte)` sur le modèle de `classer`, avec un champ en plus : `"dollars"`. Récupère le texte du bloc dont le `type` vaut `"text"`. Si `reponse.stop_reason == "refusal"`, mets `prediction` à `None`.
2. **Pilote sur 20 avis** : accuracy, latence, coût total. **Estime le coût pour 500 et pour 20 000 avis** avant d'aller plus loin.
3. Si le coût te convient : les 500 avis, sauvegardés dans `data/claude_500.csv`, puis accuracy et marge d'erreur.
4. Ajoute une colonne Claude au tableau des avis pièges du 4.6.

In [ ]:
# À TOI


**Mes résultats (4.7)** : ...

## 4.8 Le bilan

C'est **le tableau central de ton projet** : il ira dans le README et tu pourras le présenter en entretien.

### À TOI (4.8)
Complète le tableau avec tes mesures. Les cases « étape 5 » seront remplies plus tard, quand on mesurera la vitesse de TF-IDF et de CamemBERT sur ton PC.

| | TF-IDF + bigrammes | CamemBERT | Qwen 2.5 3B (local) | Claude (API) |
|---|---|---|---|---|
| Accuracy | 93,8 % (20 000 avis) | 95,8 % (20 000 avis) | ... (500 avis, ± ...) | ... |
| Avis d'entraînement | 160 000 | 20 000 | 0 | 0 |
| Entraînement | quelques minutes, CPU | ~4 min, GPU | aucun | aucun |
| Latence par avis | étape 5 | étape 5 | ... s | ... s |
| Coût pour 1 000 avis | ~0 $ (ton PC) | ~0 $ (ton PC) | ~0 $ (ton PC) | ... $ |
| Données envoyées à l'extérieur | non | non | non | oui (API) |

**Ma conclusion (4.8)** : quel modèle choisirais-tu pour classer **un million d'avis par jour** ? Et pour un prototype à livrer **demain**, sans aucune donnée étiquetée ?